In [5]:
import pandas as pd
import numpy as np

def load_and_prepare_credit_data(app_path='application_record.csv', credit_path='credit_record.csv'):
    """
    Funzione per caricare e preparare i dati in modo riproducibile.
    Utilizza un approccio flessibile per gestire CSV con separatori diversi (virgola o punto e virgola).
    """
    
    # 1. Caricamento resiliente ai separatori
    # Cerchiamo prima di leggere con la virgola. Se il dataset ha 1 sola colonna (perché tutto
    # è schiacciato insieme dal punto e virgola), ricarichiamo usando il punto e virgola.
    app_df = pd.read_csv(app_path)
    if app_df.shape[1] == 1:
        app_df = pd.read_csv(app_path, sep=';')
        
    credit_df = pd.read_csv(credit_path)
    if credit_df.shape[1] == 1:
         credit_df = pd.read_csv(credit_path, sep=';')

    # Controllo di sicurezza: verifichiamo che la colonna 'ID' esista
    if 'ID' not in app_df.columns:
        # Se 'ID' non c'è ancora, la formattazione è compromessa a un livello che Python non può gestire in automatico.
        raise ValueError("ERRORE CRITICO: Impossibile identificare la colonna 'ID' in application_record.csv. Controlla la formattazione del file grezzo.")
    if 'ID' not in credit_df.columns:
        raise ValueError("ERRORE CRITICO: Impossibile identificare la colonna 'ID' in credit_record.csv. Controlla la formattazione del file grezzo.")

    print(f"Dimensioni originali - Anagrafica: {app_df.shape}, Credito: {credit_df.shape}")

    # 2. Creazione della Variabile Target
    # 2,3,4,5 indicano ritardi oltre i 60 giorni.
    credit_df['IS_DEFAULT_MONTH'] = credit_df['STATUS'].apply(lambda x: 1 if x in ('2', '3', '4', '5') else 0)
    target_df = credit_df.groupby('ID')['IS_DEFAULT_MONTH'].max().reset_index()
    target_df.rename(columns={'IS_DEFAULT_MONTH': 'TARGET'}, inplace=True)

    # 3. Join
    df = pd.merge(app_df, target_df, on='ID', how='inner')

    # 4. Feature Selection & Engineering
    features_to_keep = [
        'TARGET', 'CODE_GENDER', 'FLAG_OWN_CAR', 'AMT_INCOME_TOTAL', 
        'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
        'DAYS_BIRTH', 'DAYS_EMPLOYED'
    ]
    
    # Assicuriamoci che tutte le feature richieste esistano nel dataset per evitare KeyError
    missing_features = [f for f in features_to_keep if f not in df.columns]
    if missing_features:
         raise KeyError(f"Mancano queste colonne necessarie nel dataset: {missing_features}")

    df_clean = df[features_to_keep].copy()

    # Trasformazione logica: giorni in anni. Arrotondiamo per pulizia.
    df_clean['AGE_YEARS'] = np.round(df_clean['DAYS_BIRTH'] / -365, 1)
    df_clean['EMPLOYED_YEARS'] = np.round(df_clean['DAYS_EMPLOYED'] / -365, 1)

    df_clean.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'], inplace=True)

    # Gestione anomalie: pensionati (valori positivi assurdi in DAYS_EMPLOYED) diventano 0 anni di impiego
    df_clean['EMPLOYED_YEARS'] = df_clean['EMPLOYED_YEARS'].apply(lambda x: 0 if x < 0 else x)

    # 5. Encoding
    df_final = pd.get_dummies(df_clean, drop_first=True)

    # 6. Pulizia nomi colonne per XGBoost
    # XGBoost non digerisce spazi o caratteri speciali ('<', '>', ecc.) nei nomi delle colonne derivanti dal One-Hot.
    import re
    df_final.columns = [re.sub(r'[<>, ]', '_', col) for col in df_final.columns]

    return df_final

if __name__ == "__main__":
    try:
        final_dataset = load_and_prepare_credit_data()
        final_dataset.to_csv('credit_risk_clean.csv', index=False)
        
        print("\n--- STATISTICHE DATASET FINALE ---")
        print(f"Righe: {final_dataset.shape[0]} | Colonne: {final_dataset.shape[1]}")
        print("\nDistribuzione del Target (Default):")
        print(final_dataset['TARGET'].value_counts(normalize=True) * 100)
        print("\nSalvataggio di 'credit_risk_clean.csv' completato.")
        
    except Exception as e:
        print(f"\nErrore durante l'esecuzione: {e}")

Dimensioni originali - Anagrafica: (438557, 18), Credito: (1048575, 3)

--- STATISTICHE DATASET FINALE ---
Righe: 36457 | Colonne: 18

Distribuzione del Target (Default):
TARGET
0    98.310338
1     1.689662
Name: proportion, dtype: float64

Salvataggio di 'credit_risk_clean.csv' completato.
